![image.png](https://i.imgur.com/a3uAqnb.png)

# RAG System Implementation - Homework Assignment

In this homework, you will implement a **Retrieval-Augmented Generation (RAG) system** for question answering. This project will help you understand the fundamentals of document retrieval, embedding systems, and how to combine retrieval with language models for improved performance.

## 📌 Project Overview
- **Task**: Build a RAG system to answer questions based on retrieved context
- **Dataset**: RAG Dataset 12000 from neural-bridge
- **Baseline Performance**: 33.00% accuracy with basic chunking and small LLM
- **Goal**: Either replicate the baseline OR achieve better performance (aim for >30%)

## 📚 Learning Objectives
By completing this assignment, you will:
- Understand document chunking and retrieval strategies
- Learn about embedding models and vector similarity search
- Implement FAISS vector stores for efficient retrieval
- Practice combining retrieval with language generation
- Explore optimization techniques for RAG systems
- Understand evaluation challenges in generative AI systems

## ⚠️ Important Constraints
- **No model fine-tuning allowed** - use pre-trained models only
- **Maximum model size**: 300M parameters for the generation model
- **Goal**: Either achieve baseline performance (30.00%) OR exceed it
- **Note**: You can choose to implement the exact baseline OR create your own improved version

## 🎯 Evaluation Methodology Note

**Important**: The accuracy calculation in this assignment uses **cosine similarity between generated and ground truth answers** with a 0.5 threshold. This method has several limitations:

1. **Semantic vs. Exact Matching**: Cosine similarity captures semantic meaning but may miss exact factual correctness
2. **Threshold Sensitivity**: The 0.5 threshold is arbitrary - some correct answers might score below it
3. **Length Bias**: Very short answers might have inflated similarity scores
4. **Paraphrasing Issues**: Correct paraphrases might score lower than expected
5. **False Positives**: Semantically similar but factually incorrect answers might score high

**Why We Use This Method**: Despite limitations, cosine similarity provides a reasonable approximation for semantic correctness at scale, which is more practical than manual evaluation of 2,400+ question-answer pairs.

**Real-World Note**: Production RAG systems typically use multiple evaluation metrics including exact match, BLEU scores, human evaluation, and domain-specific correctness checks.

In [ ]:
# !pip install faiss-cpu
# !pip install -U langchain-community
# !pip install bitsandbytes

## 1️⃣ Initial Setup and Environment

**Task**: Set up the required libraries and check GPU availability.

**Requirements**:
- Import all necessary libraries for RAG implementation
- Check CUDA availability for GPU acceleration
- Set up proper logging and warning suppression

In [ ]:
import torch
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
    pipeline
)
from datasets import load_dataset
import pandas as pd
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema import Document
import numpy as np
from tqdm import tqdm
import gc
import warnings
import os
import logging
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Suppress warnings and logs
warnings.filterwarnings('ignore')
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
logging.getLogger("transformers").setLevel(logging.ERROR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2️⃣ Dataset Loading and Exploration

**Task**: Load the RAG dataset and explore its structure.

**Requirements**:
- Load the neural-bridge/rag-dataset-12000 dataset
- Convert to pandas DataFrame for easier manipulation
- Analyze the dataset structure and create a large document for retrieval
- Display basic statistics about the dataset

In [ ]:
print("Loading RAG Dataset 12000...")
dataset = load_dataset("neural-bridge/rag-dataset-12000", split="test")

df = pd.DataFrame(dataset)
print(f"Dataset loaded with {len(df)} examples")

# TODO: Explore the dataset structure
# TODO: Look at sample questions, answers, and contexts
# TODO: Analyze the distribution of answer lengths

all_contexts = df['context'].unique()
large_document = "\n\n".join(all_contexts)
print(f"Combined document created with {len(large_document):,} characters")

## 3️⃣ Text Chunking and Vector Store Setup

**Task**: Implement document chunking and create a vector store for retrieval.

**Options**:
- **Option A**: Use baseline parameters (chunk_size=124, overlap=10)
- **Option B**: Experiment with your own chunking strategy

**Requirements**:
- Choose and implement a text chunking strategy
- Set up embedding model for vectorization
- Create FAISS vector store from document chunks

In [ ]:
# TODO: Choose your chunking strategy
# Option A: Baseline approach
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=124,
    chunk_overlap=10,
    length_function=len,
)

# Option B: Your custom approach
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=???,  # TODO: Experiment with different sizes
#     chunk_overlap=???,  # TODO: Experiment with overlap
#     length_function=len,
# )

print("Setting up text chunking and embeddings...")

chunks = text_splitter.split_text(large_document)
documents = [Document(page_content=chunk) for chunk in chunks]

# TODO: Choose your embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v1",  # Baseline choice
    model_kwargs={'device': 'cuda'}
)

vectorstore = FAISS.from_documents(documents, embeddings)
print("Vector store created successfully!")
print(f"Number of chunks: {len(chunks)}")

## 4️⃣ Language Model Selection and Setup

**Task**: Choose and set up a small language model for answer generation.

**Options**:
- **Option A**: Use baseline model (google/flan-t5-base)
- **Option B**: Choose your own model (<300M parameters)

**Baseline Configuration**:
- Model: google/flan-t5-base (~250M parameters)
- Temperature: 0.3
- Max length: 100 tokens

In [ ]:
# TODO: Choose your language model approach

# Option A: Baseline model
print("Loading small LLM for RAG system...")
small_model_name = "google/flan-t5-base"
small_tokenizer = AutoTokenizer.from_pretrained(small_model_name)

small_model = AutoModelForSeq2SeqLM.from_pretrained(
   small_model_name,
   torch_dtype=torch.float16,
   device_map="auto"
)

small_llm_pipeline = pipeline(
   "text2text-generation",
   model=small_model,
   tokenizer=small_tokenizer,
   torch_dtype=torch.float16,
   device_map="auto",
   do_sample=True,
   temperature=0.3,
   max_length=100
)

print("Small LLM pipeline created!")

# Option B: Your custom model
# TODO: Implement your own model selection
# small_model_name = "your-chosen-model"
# TODO: Set up your model pipeline

## 5️⃣ Evaluation Framework Setup

**Task**: Set up the evaluation framework using cosine similarity-based assessment.

**Evaluation Method Explanation**:
Our evaluation uses cosine similarity between generated answers and ground truth with a 0.5 threshold. This approach:
- ✅ Captures semantic similarity between answers
- ✅ Works at scale without manual evaluation
- ❌ May miss exact factual correctness
- ❌ Threshold (0.5) is somewhat arbitrary
- ❌ Can give false positives for semantically similar but wrong answers

**Requirements**:
- Load sentence transformer for answer comparison
- Implement similarity-based evaluation function

In [ ]:
print("Loading sentence transformer model for evaluation...")
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')  # Fast and reliable
print("Model loaded!")

def evaluate_answer_correctness(generated_answer, ground_truth, threshold=0.5):
    """
    Evaluate answer correctness using cosine similarity.

    Limitations:
    - May not capture exact factual accuracy
    - Threshold is somewhat arbitrary
    - Short answers may have inflated scores
    - Paraphrases might score lower than expected
    """
    gen_clean = generated_answer.strip().lower()
    truth_clean = ground_truth.strip().lower()

    gen_embedding = sentence_model.encode([gen_clean])
    truth_embedding = sentence_model.encode([truth_clean])

    similarity = cosine_similarity(gen_embedding, truth_embedding)[0][0]
    is_correct = similarity >= threshold

    return is_correct, similarity

## 6️⃣ RAG Implementation

**Task**: Implement your RAG question-answering system.

**Baseline Approach** (if replicating):
- Retrieve top-2 most similar chunks
- Concatenate chunks as context
- Use prompt: "Answer the question based on the context. Context: {context} Question: {question}"
- Generate with temperature=0.3, max_length=50

**Your Approach** (if improving):
- Experiment with different retrieval strategies (k value, similarity thresholds)
- Try different prompt engineering approaches
- Consider context processing improvements
- Optimize generation parameters

In [ ]:
# TODO: Implement your RAG function

def rag_answer_question(question, vectorstore, k=2):
    """
    Generate an answer using RAG approach.

    Args:
        question: Input question
        vectorstore: FAISS vector store for retrieval
        k: Number of chunks to retrieve

    Returns:
        Generated answer string
    """
    # TODO: Implement retrieval
    relevant_docs = vectorstore.similarity_search(question, k=k)
    context = " ".join([doc.page_content for doc in relevant_docs])

    # TODO: Engineer your prompt
    prompt = f"Answer the question based on the context. Context: {context} Question: {question}"

    # TODO: Generate answer
    result = small_llm_pipeline(
        prompt,
        max_length=50,  # TODO: Experiment with this
        do_sample=True,
        temperature=0.3  # TODO: Experiment with this
    )

    return result[0]['generated_text'].strip()

# Test your RAG system
test_question = df.iloc[0]['question']
test_answer = rag_answer_question(test_question, vectorstore)
print(f"Test Question: {test_question}")
print(f"Generated Answer: {test_answer}")
print(f"Ground Truth: {df.iloc[0]['answer']}")

## 7️⃣ Comprehensive Evaluation

**Task**: Evaluate your RAG system on the complete dataset.

**Baseline Performance to Match/Beat**:
- **Total Questions**: 2,399
- **Correct Answers**: 720  
- **Overall Accuracy**: 30.00%
- **Average Similarity**: 0.380
  - NOTE: some of the questions and answers were null, so be careful!!!
  - NOTE: The baseline might be a bit different for you as an LLM is not deterministic
    
**Requirements**:
- Run evaluation on the full dataset
- Track both accuracy and average similarity
- Handle errors gracefully
- Display progress and final results

In [ ]:
def run_evaluation(dataset_df, vectorstore, num_samples=None, similarity_threshold=0.5):
    """
    Run comprehensive evaluation of RAG system.

    Note: This evaluation uses cosine similarity which has limitations
    for measuring true correctness, but provides a reasonable approximation.
    """
    if num_samples is None:
        sample_df = dataset_df
    else:
        sample_df = dataset_df.sample(n=min(num_samples, len(dataset_df)), random_state=42)

    results = []
    correct_count = 0
    total_similarity = 0

    print(f"Evaluating on {len(sample_df)} samples...")

    for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
        question = row['question']
        ground_truth = row['answer']

        try:
            # Skip if ground_truth is None or NaN
            if ground_truth is None or (isinstance(ground_truth, float) and pd.isna(ground_truth)):
                continue

            generated_answer = rag_answer_question(question, vectorstore)

            # Skip if generated_answer is None
            if generated_answer is None:
                continue

            is_correct, similarity = evaluate_answer_correctness(generated_answer, ground_truth, similarity_threshold)

            if is_correct:
                correct_count += 1

            total_similarity += similarity

            results.append({
                'question': question,
                'generated_answer': generated_answer,
                'ground_truth': ground_truth,
                'similarity_score': similarity,
                'correct': is_correct
            })

        except Exception as e:
            print(f"Skipping item due to error: {e}")
            continue

    final_accuracy = correct_count / len(results) if len(results) > 0 else 0
    avg_similarity = total_similarity / len(results) if len(results) > 0 else 0

    return results, final_accuracy, avg_similarity

# Run your evaluation
print("Starting RAG evaluation...")
evaluation_results, accuracy, avg_similarity = run_evaluation(df, vectorstore)

In [ ]:

print(f"\nFINAL RESULTS")
print(f"{'='*50}")
print(f"Total Questions Evaluated: {len(evaluation_results)}")
print(f"Correct Answers: {sum(r['correct'] for r in evaluation_results)}")
print(f"Overall Accuracy: {accuracy:.2%}")
print(f"Average Similarity Score: {avg_similarity:.3f}")
print(f"{'='*50}")

## 📏 Evaluation Criteria

Your homework will be evaluated based on:

1. **Implementation Quality (40%)**
  - Correct RAG system implementation
  - Proper use of embedding models and vector stores
  - Valid model selection within constraints (<300M params)
  - Code quality and documentation

2. **Performance & Analysis (35%)**
  - Achieving reasonable performance (aim for >35% accuracy)
  - Thoughtful approach to optimization (if attempted)
  - Understanding of system components and their trade-offs
  - Bonus for beating baseline (30.00%)

3. **Code Quality & Documentation (25%)**
  - Clean, readable, and well-structured code
  - Proper error handling and edge case management
  - Clear variable naming and function documentation
  - Efficient implementation and resource management